# Element distribution

Count element occurrences in training and generated structures and plot heatmaps.
Set the data and output directories before running the analysis.


In [ ]:
import csv
import os
import warnings
warnings.filterwarnings("ignore")
from pymatgen.core import Structure
from collections import Counter

In [ ]:
def get_element_frequencies_in_cifs(
        directory: str = "."
):
    """
    Calculate element occurrence frequencies across CIF files in a directory.

    Args:
        directory (str): Directory path; defaults to the current directory.

    Returns:
        dict: Element-frequency dictionary: symbols (str) map to percentages (float).
    """
    element_counter = Counter()
    total_elements = 0

    # Iterate over all .cif files in the directory.
    for file in os.listdir(directory):
        if file.endswith(".cif"):
            try:
                # Read the crystal structure from the CIF file.
                structure = Structure.from_file(os.path.join(directory, file))
                # Count occurrences of each element.
                for element in structure.composition:
                    count = structure.composition[element]
                    element_counter[element.symbol] += count
                    total_elements += count
            except Exception as e:
                print(f"Error processing {file}: {e}")

    # Calculate element frequencies as percentages.
    element_frequencies = {
        element: (count / total_elements * 100)
        for element, count in element_counter.items()
    }

    return element_frequencies

In [ ]:
work_dir = "C:/Users/13701/Desktop/gm_cifs/cdvae/v32/to_plot/"
dataset_name = "5sets_data"
plot_data_dir = os.path.join(work_dir, dataset_name)
plot_data = get_element_frequencies_in_cifs(plot_data_dir)
print(plot_data)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_element_heatmap(element_frequencies):
    """
    Plot element frequencies as a periodic-table heatmap.
    
    Args:
        element_frequencies (dict): Element-frequency dictionary: symbols map to percentages (float).
    """
    # Periodic-table layout.
    periodic_table_layout = [
        ["H",  None, None, None, None, None, None, None, None, None, None, None, None, "He"],
        ["Li", "Be", None, None, None, None, None, None, None, None, None, "B",  "C",  "N",  "O",  "F",  "Ne"],
        ["Na", "Mg", None, None, None, None, None, None, None, None, None, "Al", "Si", "P",  "S",  "Cl", "Ar"],
        ["K",  "Ca", "Sc", "Ti", "V",  "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Ga", "Ge", "As", "Se", "Br", "Kr"],
        ["Rb", "Sr", "Y",  "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd", "In", "Sn", "Sb", "Te", "I",  "Xe"],
        ["Cs", "Ba", None, "Hf", "Ta", "W",  "Re", "Os", "Ir", "Pt", "Au", "Hg", "Tl", "Pb", "Bi", "Po", "At", "Rn"],
        ["Fr", "Ra", None, "Rf", "Db", "Sg", "Bh", "Hs", "Mt", "Ds", "Rg", "Cn", "Nh", "Fl", "Mc", "Lv", "Ts", "Og"]
    ]

    # Calculate the number of rows and maximum number of columns.
    num_rows = len(periodic_table_layout)
    num_cols = max(len(row) for row in periodic_table_layout)

    # Initialize heatmap data.
    heatmap_data = np.zeros((num_rows, num_cols))
    element_positions = {}

    # Populate element positions and frequencies.
    for i, row in enumerate(periodic_table_layout):
        for j, element in enumerate(row):
            if element is not None:
                element_positions[element] = (i, j)
                heatmap_data[i, j] = element_frequencies.get(element, 0)

    # Create the heatmap.
    plt.figure(figsize=(15, 8))
    ax = sns.heatmap(
        heatmap_data,
        annot=False,
        cmap="YlGnBu",
        cbar_kws={"label": "Frequency (%)"},
        linewidths=0.5,
        linecolor="white",
        xticklabels=False,
        yticklabels=False,
    )

    # Add element symbols to the heatmap.
    for element, (i, j) in element_positions.items():
        plt.text(j + 0.5, i + 0.5, element, ha="center", va="center", color="black")

    # Set the title.
    plt.title("Element Frequency Heatmap (Periodic Table Layout)", fontsize=16)
    plt.xlabel("")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_element_heatmap(element_frequencies=plot_data)

In [ ]:
from mendeleev import element
import matplotlib as mpl
import matplotlib.patches as patches


def plot_periodic_table_heatmap(plot_data):
    """
    Plot a periodic-table heatmap.

    Args:
        plot_data (dict): Element data: symbols (str) map to heatmap values (int/float).
    """
    # Configure plotting parameters.
    cell_length = 1
    cell_gap = 0.1
    cell_edge_width = 0.5

    # Prepare element data.
    elements = []
    for i in range(1, 119):
        ele = element(i)
        ele_group, ele_period = ele.group_id, ele.period

        # Handle the group and period positions of the lanthanide and actinide series.
        if 57 <= i <= 71:
            ele_group = i - 57 + 3
            ele_period = 8
        if 89 <= i <= 103:
            ele_group = i - 89 + 3
            ele_period = 9

        elements.append([i, ele.symbol, ele_group, ele_period, plot_data.get(ele.symbol, 0)])

    # Add labels for the lanthanide and actinide series.
    elements.append([None, 'LA', 3, 6, None])
    elements.append([None, 'AC', 3, 7, None])
    elements.append([None, 'LA', 2, 8, None])
    elements.append([None, 'AC', 2, 9, None])

    # Initialize the figure.
    fig = plt.figure(figsize=(10, 5))
    xy_length = (20, 11)  # x- and y-axis limits.

    # Get the colormap.
    my_cmap = plt.cm.YlOrRd  # Use plt.cm directly.
    norm = mpl.colors.Normalize(1, 100)  # Adjust the lower and upper limits as needed.
    my_cmap.set_under('None')  # Set the color below the lower limit to transparent.
    cmmapable = mpl.cm.ScalarMappable(norm=norm, cmap=my_cmap)

    # Draw the colorbar.
    plt.colorbar(cmmapable, drawedges=False)

    # Draw the periodic table.
    for e in elements:
        ele_number, ele_symbol, ele_group, ele_period, ele_count = e

        if ele_group is None:
            continue

        # Calculate the cell position.
        x = (cell_length + cell_gap) * (ele_group - 1)
        y = xy_length[1] - ((cell_length + cell_gap) * ele_period)

        # Increase spacing for the lanthanide and actinide series.
        if ele_period >= 8:
            y -= cell_length * 0.5

        # Draw the cell and fill its color.
        if ele_number:
            fill_color = my_cmap(norm(ele_count))
            rect = patches.Rectangle(
                xy=(x, y), width=cell_length, height=cell_length,
                linewidth=cell_edge_width, edgecolor='k', facecolor=fill_color
            )
            plt.gca().add_patch(rect)

        # Add the atomic number.
        plt.text(
            x + 0.04, y + 0.8, ele_number,
            va='center', ha='left',
            fontdict={'size': 6, 'color': 'black', 'family': 'Helvetica'}
        )

        # Add the element symbol.
        plt.text(
            x + 0.5, y + 0.5, ele_symbol,
            va='center', ha='center',
            fontdict={'size': 9, 'color': 'black', 'family': 'Helvetica', 'weight': 'bold'}
        )

        # Add the heatmap value.
        if ele_count is not None:
            plt.text(
                x + 0.5, y + 0.12, f"{ele_count:.1f}",
                va='center', ha='center',
                fontdict={'size': 6, 'color': 'black', 'family': 'Helvetica'}
            )

    # Set figure properties.
    plt.axis('equal')  # Keep cells square.
    plt.axis('off')  # Hide the axes.
    plt.tight_layout()  # Adjust the layout automatically.
    plt.ylim(0, xy_length[1])
    plt.xlim(0, xy_length[0])
    plt.show()

In [ ]:
plot_periodic_table_heatmap(plot_data)

In [ ]:
from matplotlib import cm
import mendeleev

cell_length = 1
# Spacing between cells.
cell_gap = 0.1
# Cell border width.
cell_edge_width = 0.5

# Get each element's atomic number, period (row), group (column), and plotting value; use zero for missing values.
elements = []
for i in range(1, 119):
    ele = mendeleev.element(i)
    ele_group, ele_period = ele.group_id, ele.period

    # Place the lanthanide series in row 8.
    if 57 <= i <= 71:
        ele_group = i - 57 + 3
        ele_period = 8
    # Place the actinide series in row 9.
    if 89 <= i <= 103:
        ele_group = i - 89 + 3
        ele_period = 9

    elements.append([i, ele.symbol, ele_group, ele_period,
                     plot_data.setdefault(ele.symbol, 0)])

# Set labels for the lanthanide and actinide series.
elements.append([None, 'LA', 3, 6, None])
elements.append([None, 'AC', 3, 7, None])
elements.append([None, 'LA', 2, 8, None])
elements.append([None, 'AC', 2, 9, None])

# Create a Matplotlib figure.
fig = plt.figure(figsize=(10, 5))
# x- and y-axis limits.
xy_length = (20, 11)

# Get the YlOrRd colormap.
# my_cmap = cm.get_cmap('YlOrRd')
my_cmap = plt.cm.YlOrRd
# Map plot_data values to colors; adjust the scale as needed.
norm = mpl.colors.Normalize(1, 100)
# Set the color below the lower limit (None leaves it white).
my_cmap.set_under('None')
# Associate the colorbar with the scalar mapping.
cmmapable = cm.ScalarMappable(norm=norm, cmap=my_cmap)
cmmapable.set_array([])
# Draw the colorbar.
plt.colorbar(cmmapable, drawedges=False)

# Draw periodic-table cells and fill their values and colors.
for e in elements:
    ele_number, ele_symbol, ele_group, ele_period, ele_count = e
    print(ele_number, ele_symbol, ele_group, ele_period, ele_count)

    if ele_group is None:
        continue

    # Locate each cell using its x and y coordinates.
    x = (cell_length + cell_gap) * (ele_group - 1)
    y = xy_length[1] - ((cell_length + cell_gap) * ele_period)

    # Increase the separation of the lanthanide and actinide series from the main table.
    if ele_period >= 8:
        y -= cell_length * 0.5

    # Draw the border and heatmap fill when the cell atomic number is not None.
    # Do not draw borders or color fills for the lanthanide and actinide series labels.
    if ele_number:
        fill_color = my_cmap(norm(ele_count))
        rect = patches.Rectangle(xy=(x, y),
                                 width=cell_length, height=cell_length,
                                 linewidth=cell_edge_width,
                                 edgecolor='k',
                                 facecolor=fill_color)
        plt.gca().add_patch(rect)

    # Add the atomic number to the cell.
    plt.text(x + 0.04, y + 0.8,
             ele_number,
             va='center', ha='left',
             fontdict={'size': 6, 'color': 'black', 'family': 'Helvetica'})
    # Add the element symbol to the cell.
    plt.text(x + 0.5, y + 0.5,
             ele_symbol,
             va='center', ha='center',
             fontdict={'size': 9, 'color': 'black', 'family': 'Helvetica', 'weight': 'bold'})
    # Add the heatmap value to the cell.
    plt.text(x + 0.5, y + 0.12,
             ele_count,
             va='center', ha='center',
             fontdict={'size': 6, 'color': 'black', 'family': 'Helvetica'})

# Use equal x and y scales (1:1) to display square cells.
plt.axis('equal')
# Hide the axes.
plt.axis('off')
# Trim blank margins.
plt.tight_layout()
# Set the x- and y-axis limits.
plt.ylim(0, xy_length[1])
plt.xlim(0, xy_length[0])

# Save the figure in SVG vector format.
# plt.savefig('C:/Users/13701/Desktop/oxyf_periodic_table_percent.svg')
# Show the figure.
plt.show()